<style>
/* Limit text outputs for all cells */
div.output_area pre {
    max-height: 300px;
    overflow: auto;
}
/* Limit rich outputs */
div.output_area div.output_subarea {
    max-height: 300px;
    overflow: auto;
}
</style>

# Deep PheWAS Step 1
## 0. Set the working directory
**ALWAYS DO THIS AT THE START OF THE SESSION BEFORE RUNNING ANY CELLS BELOW**


In [ ]:
cd /opt/notebooks/deep_phewas_RAP_install

If you want to close the jupyter session and resume the workflow later remember to `Create Snapshot` from the `DNAnexus` menu and then start your new jupyter lab session specifying the saved snapshot to load. 

Snapshots are saved in the root of your project under `/.Notebook_snapshots` with the date in the filename.
## 1. Set up the environment
###    a) Install Java
Java is required for the WDL compiler.

In [ ]:
apt update
apt install -y openjdk-8-jre

### b) Install R tidyverse
The Deep PheWAS R install scripts use tidyverse.

In [ ]:
Rscript -e 'install.packages("tidyverse")'

## 2. Install the WDL workflows to your RAP project

### a) Set the RAP directory where you will install Deep PheWAS
Edit the `options.config` file and set `PROJECT_DIR` e.g.
```bash
PROJECT_DIR=/deep_phewas
```

### b) Run the install_workflows.sh script
This script will:
* Create a docker image in your project containing the DeepPheWAS package, plink2 and dependencies.
* Download DNANexus dxCompiler required to compile WDL workflows to DNANexus workflows to run on the RAP.
* Compile and install the WDL DeepPheWAS workflows in your RAP project.

In [13]:
./install_workflows.sh

extraOptions.json not found, running update_docker.sh
Selected project project-Gbb58ZjJv6Z9G2gzB1436228
DeepPheWAS.docker.tar.gz
fields-minimum.txt.gz
sha256:3463037e123aaa424ec385d672b4230b3843006288fa2af06113ae762018677d
Preparing to copy...Copying from container - 0BSuccessfully copied 3.58kB to /opt/notebooks/deep_phewas_RAP_install/.
69f5266eb55d74959c124f77f2919c0bba4f3a8a4391a6d4a743bfd88fab72e1
file-J4pyFXjJv6Z74FF2Q1xZPfXf
Compiling WDL
workflow-J4pyFqQJv6Z1P2kb6Vx04FJ6
applet-J4pyFy0Jv6ZB4XG9kvg4Kkyx
workflow-J4pyGG8Jv6ZKg6GP3V1xqpbG
workflow-J4pyGVQJv6ZK201qFBQGg6QG


## 3. Extracting the UK Biobank data fields that provide the input for phenotype generation
### a) Find the most up to date UK Biobank phenotype data

In [15]:
dx ls -l /*.dataset

closed  2025-07-22 13:47:19           app648_20250722114356.dataset (record-J1zkGk8JpQJYGkZzY7kvP7p7)
closed  2023-12-07 23:50:21           app648_20231207195939.dataset (record-Gbk5ZKQJpQJZQg52Z4FVY7k1)


Edit options.config and set the data set to the latest version above e.g. `DATASET=/app648_20250722114356.dataset`
### b) Extract the phenotype data from the most up-to-date source
Run `./extract_fields.sh`, which will launch several RAP jobs to extract:
* Participant data fields
* Hospital episode statistics (HES)
* Death registry data
* Primary care data

Wait for these jobs to finish.


In [16]:
./extract_fields.sh

fields-minimum.txt.gz
app648_20250722114356.dataset
[===========================================================>] Uploaded 21,503 of 21,503 bytes (100%) fields_use.txt
ID                                file-J4pyfVjJv6ZGqB2XKy4PQKqG
Class                             file
Project                           project-Gbb58ZjJv6Z9G2gzB1436228
Folder                            /deep_phewas/inputs
Name                              fields_use.txt
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Sun Dec  7 21:01:59 2025
Created by                        nshrine
 via the job                      job-J4pv050Jv6Z5272fgFBzjX74
Last modified                     Sun Dec  7 21:02:00 2025
Media type                        
archivalState                     "live"
cloudAccount                   

## 4. Phenotype generation
### a) Make the configuration file specifying the inputs for phenotype generation
Run `make_inputs_phenotype_generation.sh` which will make the .json configuration file specifying the inputs required for the phenotype generation step.

**Ignore warnings** `missing input for non-optional parameter`

`Intermediate representation` means it has run correctly.

In [17]:
./make_inputs_phenotype_generation.sh

[warning] missing input for non-optional parameter files
[warning] missing input for non-optional parameter save_loc
[warning] missing input for non-optional parameter save_loc
[warning] missing input for non-optional parameter min_data
[warning] missing input for non-optional parameter GPC
[warning] missing input for non-optional parameter GPP
[warning] missing input for non-optional parameter hesin_diag
[warning] missing input for non-optional parameter HESIN
[warning] missing input for non-optional parameter hesin_oper
[warning] missing input for non-optional parameter death
[warning] missing input for non-optional parameter death_cause
[warning] missing input for non-optional parameter health_data
[warning] missing input for non-optional parameter sex_info
[warning] missing input for non-optional parameter min_data
[warning] missing input for non-optional parameter GPP
[warning] missing input for non-optional parameter health_data
[warning] missing input for non-optional parameter 

### b) Run the phenotype generation
Run `run_phenotype_generation.sh` which will submit a RAP analysis pipeline to generate the phenotypes.

The pipeline submits several sub-jobs that run in parallel to make:
* Data field phenotypes
* Phecode phenotypes
* Primary care phenotypes
* Formula phenotypes
* Composite phenotypes

Wait until the pipeline has completed before proceeding.

In [18]:
./run_phenotype_generation.sh

analysis-J4pz4p0Jv6ZKy6J7KQYJF95Y


## 5. Phenotype preparation
### a) Make the configuration file specifying the inputs for phenotype preparation
In `options.config` you can change:
* Whether to remove related samples (related_remove=true/false, default=false)
* Whether to rank inverse-normal transform phenotypes (IVNT=true/false, default=true)
* Any groupings of samples, e.g. by ancestry (groupings=ancestry_panUKB)

Run `make_inputs_phenotype_preparation.sh` which will make the .json configuration file specifying the inputs required for the phenotype preparation step.

`Intermediate representation` means it has run correctly.

In [4]:
./make_inputs_phenotype_preparation.sh

Intermediate representation


### b) Run the phenotype preparation
Run `run_phenotype_preparation.sh` which will submit the RAP job to create the phenotype tables to be used in association testing.

Wait until the pipeline has completed before proceeding.

In [5]:
./run_phenotype_preparation.sh

job-J4q8Q38Jv6Z8gk7QBbg3fvjQ


## 6. Regenie Step 1
### a) Extract the covariates to use in the association testing from the UK Biobank data
In `options.config` you can change:
* The name of the covariate file to create (covar=, default=covar_regenie.txt)
* The names of the continuous covariate columns to include (covarColList=, default=age,PC{1:10})
* The names of categorical covariates to include (catCovarList=, defaults=sex,array)

Run `extract_covar.sh` and wait for the RAP job to finish.

In [19]:
./extract_covar.sh

file-J4pzY18Jv6ZB4XG9kvg4bGqz
job-J4pzY1QJv6Z74FF2Q1xZXyYY
[===========================================================>] Completed 50,431,350 of 50,431,350 bytes (100%) /opt/notebooks/deep_phewas_RAP_install/covar_extracted.csvv
Rows: 501954 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (14): eid, 31-0.0, 22000-0.0, 21022-0.0, 22009-0.1, 22009-0.2, 22009-0.3...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.



### b) Make the configuration file specifying the inputs for regenie step 1
In `options.config` you can set:
* The IDs of phenotypes to include if you only want to use a subset of all phenotypes available (phenoColList=).

Run `make_inputs_regenie_step1.sh` which will make the .json configuration file specifying the inputs required for regenie step 1.

**Ignore warnings** `missing input for non-optional parameter`

`Intermediate representation` means it has run correctly.

In [6]:
./make_inputs_regenie_step1.sh

Rows: 21211 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (1): group
dbl (1): eid

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
[warning] missing input for non-optional parameter prepared_phenotypes
[warning] missing input for non-optional parameter genos
[warning] missing input for non-optional parameter phewas_manifest
[warning] missing input for non-optional parameter missing_thresh
Intermediate representation


### Run regenie step 1
Run `run_regenie_step1.sh` which will submit a RAP workflow that runs the following tasks:
* Divides the phenotype files from the Phenotype preparation step above into binary and quantitative traits
* Clusters the traits by patterns of missingness such that no cluster has any trait with >15% missingness (value can be altered in `RAP.config`).
* For each cluster of traits a set of SNPs is filtered to have --geno 0.1 --hwe 1e-15 --mac 100 --maf 0.01 --mind 0.1

Regenie step 1 is run on each cluster.

In [8]:
./run_regenie_step1.sh

analysis-J4q8kf8Jv6Z8G74Jk44QY5fy


The output will be in ${PROJECT_DIR}/step1 and comprises:
* A set of _pred.list files 1 for each sample grouping as specified by the grouping file (usually grouped for ancestry), combined across all phenotypes.
* A set of .loco files, 1 for each sample grouping and phenotype.